# Voice lab: one recording batch, end to end (ADR-008)

You record a batch and upload it to Google Drive. This notebook does the rest:

1. **Cuts** your long recordings into one clip per line, using the line numbers you spoke.
2. **Checks** every clip against its line with speech recognition, so a skipped word shows up.
3. **Clones** each line with the speech model for its language, copying the voice from one of
   your *other* clips: **F5-TTS** for English, **IndicF5** for Hindi and Marathi.
4. Runs the clone through **your voice model** (Applio RVC).
5. **Scores** each clone against your real recording of the same line, and writes a report and
   a blind listening set to Drive.

**Round 1 note.** A batch holds only a few minutes per language. That is too little to fine-tune
F5-TTS or IndicF5, so this round clones from your clips instead. Fine-tuning comes once a language
has about 30 minutes. Your voice model is retrained with `train_colab.ipynb` once enough new audio
has built up.

**Before the first run:**

- Runtime > Change runtime type > **T4 GPU**.
- Accept the terms on https://huggingface.co/ai4bharat/IndicF5, then add a Hugging Face read
  token as a Colab secret named `HF_TOKEN` (the key icon on the left). Allow this notebook to use it.
- Then **Runtime > Run all**. Allow the Google Drive popup.

The notebook prints file names and numbers only, never your text.

## 1. Check that a GPU is attached

In [ ]:
# @title Check the GPU
import shutil
import subprocess


def gpu_name():
    if not shutil.which("nvidia-smi"):
        return None
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None


GPU = gpu_name()
if GPU is None:
    raise RuntimeError(
        "No GPU found.\n"
        "  1. Runtime > Change runtime type > select 'T4 GPU' > Save, then run this cell again.\n"
        "  2. If no GPU is available, your free GPU quota is used up. Come back later."
    )
print("GPU OK:", GPU)

## 2. Settings and Google Drive

Change `BATCH` for each new batch. Your recordings go in `clone-your-voice/batches/<BATCH>/`.

In [ ]:
# @title Settings and Google Drive
import os
import re
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/clone-your-voice"  # @param {type:"string"}
BATCH = "001"  # @param {type:"string"}
VOICE = "sachinkoli2"  # @param {type:"string"}
STEPS = 32  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
PITCH = 0  # @param {type:"integer"}

if not re.fullmatch(r"[0-9]{3}", BATCH):
    raise ValueError("BATCH must be 3 digits, for example 001.")
if not re.fullmatch(r"[a-z0-9_-]{1,32}", VOICE):
    raise ValueError("VOICE must be 1 to 32 characters: lowercase letters, digits, _ or -.")

BATCH_DIR = Path(DRIVE_DIR) / "batches" / BATCH
OUT_DIR = BATCH_DIR / "results"
CODE_DIR = Path(DRIVE_DIR) / "code"
VOICE_DIR = Path(DRIVE_DIR) / VOICE
WORK = Path("/content/lab")
WORK.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "Could not mount Google Drive ("
        + repr(exc)
        + ").\nRun this cell again and approve the popup."
    ) from exc

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        "No HF_TOKEN secret. Click the key icon on the left, add HF_TOKEN with a Hugging Face "
        "read token, switch on notebook access, then run this cell again."
    ) from exc

if not BATCH_DIR.is_dir():
    raise RuntimeError(f"{BATCH_DIR} does not exist. Upload your recordings there first.")
print("Drive OK. Batch folder:", BATCH_DIR)

## 3. Load the helper and read the batch

`code/cyv_lab.py` in your Drive folder holds the cutting, checking and scoring logic. It is a copy
of `notebooks/cyv_lab.py` from the repository.

In [ ]:
# @title Read the batch and find your recordings
import hashlib
import importlib.util
import sys
from pathlib import Path

helper = CODE_DIR / "cyv_lab.py"
if not helper.is_file():
    raise RuntimeError(f"{helper} is missing. Copy notebooks/cyv_lab.py from the repository there.")
print("cyv_lab.py sha256:", hashlib.sha256(helper.read_bytes()).hexdigest()[:16])
spec = importlib.util.spec_from_file_location("cyv_lab", helper)
lab = importlib.util.module_from_spec(spec)
sys.modules["cyv_lab"] = lab
spec.loader.exec_module(lab)

batch_file = BATCH_DIR / f"batch-{BATCH}.tsv"
if not batch_file.is_file():
    raise RuntimeError(f"{batch_file} is missing. It is the script for this batch.")
LINES = lab.parse_batch(batch_file.read_text(encoding="utf-8"))
BY_FILE = {x.file: x for x in LINES}
UP = lab.find_uploads(BATCH_DIR, BATCH, LINES)
for p in UP.problems:
    print("NOTE:", p)
if UP.missing:
    raise RuntimeError("No recording for: " + ", ".join(UP.missing))
print(
    len(LINES), "lines.", "Long recordings:", ", ".join(p.name for p in UP.takes.values()) or "none"
)
if UP.clips:
    print("Single-line files:", len(UP.clips))


def find_model(voice_dir, name):
    """Highest-epoch small model, the same rule as train_colab.ipynb (never the G_/D_ files)."""
    rx = re.compile(rf"^{re.escape(name)}_(\d+)e_(\d+)s\.pth$")
    best, best_key = None, (-1, -1)
    for folder in (voice_dir / "checkpoints", voice_dir):
        if folder.is_dir():
            for p in folder.glob("*.pth"):
                m = rx.match(p.name)
                if m and (int(m[1]), int(m[2])) > best_key:
                    best, best_key = p, (int(m[1]), int(m[2]))
    return best


def find_index(voice_dir, name):
    for folder in (voice_dir / "checkpoints", voice_dir):
        if (folder / f"{name}.index").is_file():
            return folder / f"{name}.index"
    return None


MODEL_PTH, MODEL_INDEX = find_model(VOICE_DIR, VOICE), find_index(VOICE_DIR, VOICE)
if MODEL_PTH is None or MODEL_INDEX is None:
    raise RuntimeError(f"Could not find the voice model '{VOICE}' under {VOICE_DIR}.")
print("Voice model:", MODEL_PTH.name, "|", MODEL_INDEX.name)

## 4. Cut and check the clips

Speech recognition (faster-whisper, large-v3) finds the line numbers you spoke, in English.
A line's clip runs from just after its number to the next number. If you said a number twice,
the last take is kept.

Then each clip is transcribed again in its own language and compared with its line. A clip
whose error rate is above 0.35 is flagged, and is not used as a voice to clone from.

In [ ]:
# @title Install faster-whisper
import subprocess  # noqa: F811 (each cell can run on its own)
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.2.1"], check=True)
print("faster-whisper installed.")

In [ ]:
# @title Cut every recording into clips and check them
import json
import subprocess

import numpy as np
from faster_whisper import WhisperModel

ASR_REVISION = "edaa852ec7e145841d8ffdb056a99866b5f0a478"  # Systran/faster-whisper-large-v3
asr = WhisperModel("large-v3", device="cuda", compute_type="float16", revision=ASR_REVISION)


def load16k(path):
    """Any audio file -> mono float32 at 16 kHz, through ffmpeg."""
    r = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", str(path), "-ac", "1", "-ar", "16000",
         "-f", "f32le", "pipe:1"],
        capture_output=True, check=True,
    )  # fmt: skip
    return np.frombuffer(r.stdout, np.float32)


def words(samples, lang):
    """Used only to hear the spoken line numbers, so the prompt lists number words."""
    segs, _ = asr.transcribe(samples, language=lang, word_timestamps=True, beam_size=5,
                             vad_filter=False, condition_on_previous_text=False,
                             initial_prompt=lab.NUMBER_PROMPT)  # fmt: skip
    return [lab.Word(w.word.strip(), w.start, w.end) for s in segs for w in (s.words or [])]


def text_of(samples, lang):
    segs, _ = asr.transcribe(samples, language=lang, beam_size=5, condition_on_previous_text=False)
    return " ".join(s.text.strip() for s in segs)


def write_clip(src, start, end, dst):
    """Cut [start, end] seconds from the original file into a 24 kHz mono WAV."""
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(src)]
    if start is not None:
        cmd += ["-ss", f"{start:.3f}", "-to", f"{end:.3f}"]
    subprocess.run(cmd + ["-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(dst)], check=True)


CLIPS = OUT_DIR / "clips"
CLIPS.mkdir(parents=True, exist_ok=True)
QA, SECONDS = {}, {}
groups = lab.takes(LINES)
for take, group in groups.items():
    todo = [x for x in group if x.file not in UP.clips]
    if not todo:
        continue
    src = UP.takes[take]
    cut = lab.cut_take(load16k(src), len(group), words)
    for p in cut.problems:
        print(f"NOTE {src.name}:", p)
    for n, x in enumerate(group, start=1):
        if x.file in UP.clips or n not in cut.spans:
            continue
        s, e = cut.spans[n]
        write_clip(src, s, e, CLIPS / f"{x.file}.wav")
        if cut.retakes.get(n):
            print(f"{x.file}: kept the last of {cut.retakes[n] + 1} takes")
for f, src in UP.clips.items():
    write_clip(src, None, None, CLIPS / f"{f}.wav")

for x in LINES:
    wav = CLIPS / f"{x.file}.wav"
    if not wav.is_file():
        QA[x.file] = {"ok": False, "cer": None, "why": "no clip"}
        continue
    audio = load16k(wav)
    SECONDS[x.file] = len(audio) / 16000
    if x.engine is None:  # singing: no transcript check
        QA[x.file] = {"ok": True, "cer": None, "why": "singing, not checked"}
        continue
    c = lab.cer(x.text, text_of(audio, x.lang))
    QA[x.file] = {"ok": c <= lab.QA_MAX_CER, "cer": round(c, 3)}
    flag = "" if QA[x.file]["ok"] else "  <- check this one by ear"
    print(f"{x.file}: {SECONDS[x.file]:.1f} s, error rate {c:.2f}{flag}")

(OUT_DIR / "clips.json").write_text(
    json.dumps({"qa": QA, "seconds": SECONDS}, indent=2) + "\n", encoding="utf-8"
)
print(sum(q["ok"] for q in QA.values()), "of", len(LINES), "clips passed. Saved in", CLIPS)
del asr

## 5. Install the speech models and Applio

Each one gets its own virtual environment, so they cannot break each other:

- **F5-TTS** `1.1.22` (English).
- **IndicF5** (Hindi, Marathi), from AI4Bharat's package, with the model pinned to one revision.
- **Applio** at the pinned commit, the same as `train_colab.ipynb`.

In [ ]:
# @title Install F5-TTS (own virtual environment)
import subprocess
import sys
from pathlib import Path

F5_VENV = Path("/content/venv_f5")
F5_PY = str(F5_VENV / "bin" / "python")
F5_VERSION = "f5-tts==1.1.22"


def sh(cmd, cwd=None, env=None):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, env=env, check=True)


marker = F5_VENV / ".ready"
if marker.is_file() and marker.read_text().strip() == F5_VERSION:
    print("F5-TTS already installed, skipping.")
else:
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh(["uv", "venv", F5_VENV, "--python", "3.11", "--clear"])
    sh(["uv", "pip", "install", "-q", "--python", F5_PY, F5_VERSION])
    marker.write_text(F5_VERSION)
sh([F5_PY, "-c", "import f5_tts.api, torch; print('F5 OK, CUDA:', torch.cuda.is_available())"])

In [ ]:
# @title Install IndicF5 (own virtual environment)
from pathlib import Path

INDIC_VENV = Path("/content/venv_indicf5")
INDIC_PY = str(INDIC_VENV / "bin" / "python")
INDIC_REVISION = "fb652a96e508091045890dae5bb99fe511facbcf"  # ai4bharat/IndicF5 on Hugging Face
# The same packages as AI4Bharat's own demo Space.
INDIC_PACKAGES = [
    "git+https://github.com/ai4bharat/IndicF5.git",
    "transformers<4.50",
    "torch==2.8.0",
    "torchaudio==2.8.0",
    "soundfile",
]

marker = INDIC_VENV / ".ready"
if marker.is_file() and marker.read_text().strip() == INDIC_REVISION:
    print("IndicF5 already installed, skipping.")
else:
    sh(["uv", "venv", INDIC_VENV, "--python", "3.11", "--clear"])
    sh(["uv", "pip", "install", "-q", "--python", INDIC_PY, *INDIC_PACKAGES])
    marker.write_text(INDIC_REVISION)
sh(
    [
        INDIC_PY,
        "-c",
        "import torch, transformers; print('IndicF5 env OK, CUDA:', torch.cuda.is_available())",
    ]
)

In [ ]:
# @title Install Applio at the pinned commit
import hashlib
import sys
import urllib.request
from pathlib import Path

# Must equal APPLIO_COMMIT in the repo's Applio pin file and in notebooks/train_colab.ipynb.
APPLIO_REPO = "https://github.com/IAHispano/Applio"
APPLIO_COMMIT = "c817f1f83f689ef72e0dfeb1c17ae990b9248506"

CONSTRAINTS_URL = (
    "https://raw.githubusercontent.com/kolisachint/clone-your-voice/main/notebooks/constraints.txt"
)
CONSTRAINTS_SHA256 = "b6d5e81f1e02c787276aab11835eb5b5c9e69db3b9970c93d00c2bb6b3176e40"

APPLIO = Path("/content/Applio")
VENV = Path("/content/applio-venv")
PY = str(VENV / "bin" / "python")
CONSTRAINTS = Path("/content/constraints.txt")

with urllib.request.urlopen(CONSTRAINTS_URL, timeout=60) as resp:  # noqa: S310
    blob = resp.read()
if hashlib.sha256(blob).hexdigest() != CONSTRAINTS_SHA256:
    raise RuntimeError(
        "constraints.txt does not match the expected hash. Re-open the latest notebook."
    )
CONSTRAINTS.write_bytes(blob)

marker = VENV / ".ready"
if marker.is_file() and marker.read_text().strip() == APPLIO_COMMIT:
    print("Applio already installed for this commit, skipping.")
else:
    APPLIO.mkdir(exist_ok=True)
    if not (APPLIO / ".git").is_dir():
        sh(["git", "init", "-q"], cwd=APPLIO)
        sh(["git", "remote", "add", "origin", APPLIO_REPO], cwd=APPLIO)
    sh(["git", "fetch", "--depth", "1", "origin", APPLIO_COMMIT], cwd=APPLIO)
    sh(["git", "-c", "advice.detachedHead=false", "checkout", "-q", "FETCH_HEAD"], cwd=APPLIO)
    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=APPLIO, capture_output=True, text=True, check=True
    ).stdout.strip()
    if head != APPLIO_COMMIT:
        raise RuntimeError(f"Applio is at {head}, expected {APPLIO_COMMIT}")
    sh(["apt-get", "install", "-y", "-qq", "portaudio19-dev"])
    sh(["uv", "venv", VENV, "--python", "3.12", "--seed", "--allow-existing"])
    sh(
        ["uv", "pip", "install", "-q", "-r", APPLIO / "requirements-engine.txt", "-c", CONSTRAINTS,
         "--extra-index-url", "https://download.pytorch.org/whl/cu128",
         "--index-strategy", "unsafe-best-match", "--python", PY]
    )  # fmt: skip
    sh(
        [PY, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--no-exe"],
        cwd=APPLIO,
    )
    marker.write_text(APPLIO_COMMIT)
print("Applio ready at", APPLIO_COMMIT[:10])

## 6. Clone every line

For each line, the speech model for its language copies the voice from one of your *other*
clips in that language. A clip in the same style is preferred (happy clones from happy). Then it
speaks the line, so the result can be compared with your real recording.

In [ ]:
# @title Clone each line with F5-TTS or IndicF5
import json
import os

TTS_DIR = OUT_DIR / "tts"
TTS_DIR.mkdir(parents=True, exist_ok=True)

JOBS = {"f5": [], "indicf5": []}
REF_OF = {}
for x in LINES:
    if x.engine is None or not (CLIPS / f"{x.file}.wav").is_file():
        continue
    ref = lab.pick_reference(x, LINES, QA, SECONDS)
    if ref is None:
        print(f"{x.file}: no other usable clip in {x.lang} to clone from, skipped")
        continue
    REF_OF[x.file] = ref.file
    JOBS[x.engine].append(
        {"out": str(TTS_DIR / f"{x.file}_tts.wav"), "text": x.text,
         "ref_wav": str(CLIPS / f"{ref.file}.wav"), "ref_text": ref.text}
    )  # fmt: skip

F5_SCRIPT = """
import json, sys, time
from pathlib import Path
from f5_tts.api import F5TTS

job = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
tts = F5TTS()
for j in job["items"]:
    if Path(j["out"]).exists():
        continue
    t0 = time.time()
    tts.infer(ref_file=j["ref_wav"], ref_text=j["ref_text"], gen_text=j["text"],
              show_info=lambda *a, **k: None, file_wave=j["out"], nfe_step=job["steps"],
              cfg_strength=2.0, sway_sampling_coef=-1.0, speed=1.0, remove_silence=False,
              seed=job["seed"])
    print(Path(j["out"]).name, f"{time.time() - t0:.1f} s", flush=True)
"""

INDIC_SCRIPT = """
import json, os, sys, time
from pathlib import Path
import numpy as np, soundfile as sf, torch
from transformers import AutoModel

job = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True,
                                  revision=job["revision"], token=os.environ["HF_TOKEN"])
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
for j in job["items"]:
    if Path(j["out"]).exists():
        continue
    t0 = time.time()
    audio = model(j["text"], ref_audio_path=j["ref_wav"], ref_text=j["ref_text"])
    audio = np.asarray(audio)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    sf.write(j["out"], audio.reshape(-1), 24000)
    print(Path(j["out"]).name, f"{time.time() - t0:.1f} s", flush=True)
"""

for engine, py, script, extra in (
    ("f5", F5_PY, F5_SCRIPT, {"steps": STEPS, "seed": SEED}),
    ("indicf5", INDIC_PY, INDIC_SCRIPT, {"revision": INDIC_REVISION}),
):
    if not JOBS[engine]:
        continue
    (WORK / f"run_{engine}.py").write_text(script, encoding="utf-8")
    job = WORK / f"job_{engine}.json"  # holds the text; stays on this runtime's disk
    job.write_text(json.dumps({"items": JOBS[engine], **extra}), encoding="utf-8")
    print(f"--- {engine}: {len(JOBS[engine])} line(s)")
    sh([py, WORK / f"run_{engine}.py", job], env=dict(os.environ))
print(len(list(TTS_DIR.glob("*_tts.wav"))), "cloned clips in", TTS_DIR)

## 7. Your voice on each clone (Applio RVC)

Applio loads your voice model once and converts every cloned clip.

In [ ]:
# @title Applio: your voice on each clone
import shutil  # noqa: F811 (each cell can run on its own)
import time

todo = [p for p in sorted(TTS_DIR.glob("*_tts.wav"))
        if not (TTS_DIR / p.name.replace("_tts.wav", "_tts_rvc.wav")).is_file()]  # fmt: skip
if todo:
    rvc_in, rvc_out = WORK / "rvc_in", WORK / "rvc_out"
    for d in (rvc_in, rvc_out):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True)
    for p in todo:
        shutil.copyfile(p, rvc_in / p.name)
    t0 = time.time()
    sh(
        [PY, "core.py", "batch-infer", "--input-folder", rvc_in, "--output-folder", rvc_out,
         "--pth-path", MODEL_PTH, "--index-path", MODEL_INDEX, "--pitch", PITCH,
         "--f0-method", "rmvpe", "--embedder-model", "contentvec", "--export-format", "WAV"],
        cwd=APPLIO,
    )  # fmt: skip
    for p in todo:
        produced = rvc_out / (p.stem + "_output.wav")  # Applio names its output <stem>_output.wav
        if not produced.is_file():
            raise RuntimeError(f"Applio did not write {produced.name}. Scroll up for its error.")
        shutil.copyfile(produced, TTS_DIR / p.name.replace("_tts.wav", "_tts_rvc.wav"))
    print(f"{len(todo)} clip(s) in {time.time() - t0:.0f} s")
else:
    print("Every clone is already converted.")

## 8. Score, report and blind listening set

- **Error rate** (CER): speech recognition on the clone, compared with the line. Lower is better.
- **Similarity** (0 to 1): how much the clone sounds like *your real recording of the same line*,
  from a speaker-verification model (`microsoft/wavlm-base-plus-sv`). Higher is better.

The report is `results/report.md`. The blind set is `results/blind/`: for each line, your real
clip and the two clones under shuffled letters. Listen first, then open `key.json`.

In [ ]:
# @title Score every clone and write the report
import json
import random
import shutil

import torch
from faster_whisper import WhisperModel
from transformers import AutoFeatureExtractor, WavLMForXVector

SV_REVISION = "feb593a6c23c1cc3d9510425c29b0a14d2b07b1e"  # microsoft/wavlm-base-plus-sv
fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv", revision=SV_REVISION)
sv = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv", revision=SV_REVISION)
sv = sv.to("cuda").eval()
asr = WhisperModel("large-v3", device="cuda", compute_type="float16", revision=ASR_REVISION)


def embed(samples):
    inputs = fe(samples, sampling_rate=16000, return_tensors="pt").to("cuda")
    with torch.no_grad():
        e = sv(**inputs).embeddings
    return torch.nn.functional.normalize(e, dim=-1)[0]


def sim(a, b):
    return round(float(torch.dot(embed(a), embed(b))), 3)


rows = []
for x in LINES:
    row = {"file": x.file, "engine": x.engine, "ref": REF_OF.get(x.file),
           "qa_cer": QA.get(x.file, {}).get("cer")}  # fmt: skip
    real_wav = CLIPS / f"{x.file}.wav"
    if x.engine and real_wav.is_file():
        real = load16k(real_wav)
        for key, suffix in (("tts", "_tts.wav"), ("rvc", "_tts_rvc.wav")):
            p = TTS_DIR / f"{x.file}{suffix}"
            if p.is_file():
                clone = load16k(p)
                row[f"{key}_cer"] = round(lab.cer(x.text, text_of(clone, x.lang)), 3)
                row[f"{key}_sim"] = sim(real, clone)
    rows.append(row)

(OUT_DIR / "report.json").write_text(json.dumps(rows, indent=2) + "\n", encoding="utf-8")
(OUT_DIR / "report.md").write_text(lab.report_markdown(BATCH, rows), encoding="utf-8")

blind = OUT_DIR / "blind"
shutil.rmtree(blind, ignore_errors=True)
blind.mkdir(parents=True)
rng = random.Random()  # noqa: S311  (a listening shuffle, not security)
key = {}
for x in LINES:
    options = [p for p in (CLIPS / f"{x.file}.wav", TTS_DIR / f"{x.file}_tts.wav",
                           TTS_DIR / f"{x.file}_tts_rvc.wav") if p.is_file()]  # fmt: skip
    if len(options) < 2:
        continue
    rng.shuffle(options)
    for letter, p in zip("ABC", options, strict=False):
        shutil.copyfile(p, blind / f"{x.file}_{letter}.wav")
        key[f"{x.file}_{letter}.wav"] = str(p.relative_to(OUT_DIR))
(blind / "key.json").write_text(json.dumps(key, indent=2) + "\n", encoding="utf-8")
print((OUT_DIR / "report.md").read_text(encoding="utf-8"))

## 9. Next steps

1. Tell Claude the batch is done. It reads `results/report.md` and `results/clips.json` from
   your Drive and writes up the results.
2. Listen to `results/blind/` and say which letter sounded most like you, per line.
3. Clips flagged in step 4 can be re-recorded as single-line files (`hi_conv_003.m4a`) in the same
   folder. Run the notebook again: finished work is skipped.

**Before saving this notebook to GitHub: Edit > Clear all outputs.**